<a href="https://colab.research.google.com/github/usami0jp/ai2023-old/blob/main/rag_web.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""
シンプルなRAG（検索拡張生成）デモ：Webページを読み込んで質問に答える

流れ：
  1. Webページを取得してテキストを取り出す（Retrieval の材料集め）
  2. テキストを小さな「チャンク」に分割する
  3. 各チャンクをベクトル（埋め込み）に変換する
  4. 質問もベクトルにして、似ているチャンクを探す（検索）
  5. 見つけたチャンクを資料としてLLMに渡し、回答を生成する（生成）

Google Colab でセルごとに分けて実行することを想定しています（# %% がセルの区切り）。
"""

# %% [0] ライブラリのインストール（Colabで最初に1回だけ実行）
# !pip install -q requests beautifulsoup4 sentence-transformers anthropic

# %% [1] インポート
import os
import time
import numpy as np
import requests
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer

# %% [2] 設定（ここを変えて実験してみましょう）
URLS = [
    "https://ja.wikipedia.org/wiki/転移学習",
    "https://ja.wikipedia.org/wiki/畳み込みニューラルネットワーク",
]
CHUNK_SIZE = 400   # 1チャンクの文字数
OVERLAP = 100      # 前後のチャンクで重ねる文字数（文の途中で切れる影響を減らす）
TOP_K = 3          # 検索で取り出すチャンク数


# %% [3] Webページの取得とテキスト抽出
def fetch_text(url):
    headers = {"User-Agent": "RAG-classroom-demo/1.0 (educational use)"}
    r = requests.get(url, headers=headers, timeout=15)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    # 本文以外（スクリプト、メニュー、脚注番号など）を取り除く
    for tag in soup(["script", "style", "nav", "header", "footer", "table", "sup"]):
        tag.decompose()

    # Wikipediaなら本文領域、それ以外は<main>か<body>
    main = soup.find(id="mw-content-text") or soup.find("main") or soup.body
    lines = [line.strip() for line in main.get_text(separator="\n").splitlines()]
    return "\n".join(line for line in lines if len(line) > 10)


def split_into_chunks(text, size=CHUNK_SIZE, overlap=OVERLAP):
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap
    return chunks


t = time.time()
chunks, sources = [], []
for url in URLS:
    text = fetch_text(url)
    page_chunks = split_into_chunks(text)
    chunks += page_chunks
    sources += [url] * len(page_chunks)
    print(f"{url}: {len(text)}文字 → {len(page_chunks)}チャンク")
    time.sleep(1)  # サーバーに負荷をかけないよう間隔をあける
print(f"取得にかかった時間: {time.time() - t:.1f}秒")


# %% [4] 埋め込み（ベクトル化）
# 日本語に対応した小さめの埋め込みモデル
# e5系のモデルは、文書に "passage: "、質問に "query: " を付けるのがルール
t = time.time()
embed_model = SentenceTransformer("intfloat/multilingual-e5-small")
chunk_vecs = embed_model.encode(
    ["passage: " + c for c in chunks],
    normalize_embeddings=True,   # 正規化しておくと内積＝コサイン類似度
    show_progress_bar=True,
)
print(f"ベクトルの形: {chunk_vecs.shape}")
print(f"埋め込みにかかった時間: {time.time() - t:.1f}秒")


# %% [5] 検索
def search(query, k=TOP_K):
    q_vec = embed_model.encode(["query: " + query], normalize_embeddings=True)[0]
    scores = chunk_vecs @ q_vec               # 全チャンクとの類似度
    top_idx = np.argsort(-scores)[:k]         # 類似度の高い順に k 個
    return [(float(scores[i]), chunks[i], sources[i]) for i in top_idx]


# %% [6] 生成（LLMに資料と質問を渡す）
# APIキーがあれば Claude で回答を生成。なければプロンプトを表示するだけ
# （そのプロンプトを任意のチャットAIに貼り付けても試せます）
#
# Colabの場合：左の🔑（シークレット）に ANTHROPIC_API_KEY を登録しておく
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass


def build_prompt(query, hits):
    context = "\n\n".join(
        f"[{n + 1}] (出典: {src})\n{text}" for n, (_, text, src) in enumerate(hits)
    )
    return f"""以下の資料だけを使って、質問に日本語で答えてください。
資料に答えがない場合は「資料からはわかりません」と答えてください。
根拠となる資料の番号を [1] のように付けてください。

<資料>
{context}
</資料>

質問: {query}"""


def generate(prompt):
    if not os.environ.get("ANTHROPIC_API_KEY"):
        return None
    import anthropic
    client = anthropic.Anthropic()
    msg = client.messages.create(
        model="claude-haiku-5-5",
        max_tokens=1000,
        messages=[{"role": "user", "content": prompt}],
    )
    return msg.content[0].text


def ask(query):
    hits = search(query)
    print("=== 検索結果 ===")
    for score, text, src in hits:
        print(f"[類似度 {score:.3f}] {src}\n  {text[:80].replace(chr(10), ' ')}...\n")

    prompt = build_prompt(query, hits)
    answer = generate(prompt)
    if answer is None:
        print("=== APIキーがないため、LLMに渡すプロンプトを表示します ===")
        print(prompt)
    else:
        print("=== 回答 ===")
        print(answer)


# %% [7] 質問してみる
ask("転移学習でファインチューニングとは何をすることですか？")

# %%
ask("畳み込み層とプーリング層の役割の違いは？")

# %% 資料にない質問（「わかりません」と答えるか確認）
ask("2024年のノーベル物理学賞は誰が受賞しましたか？")